# EXACT 2026 — Current Pipeline Walkthrough

Notebook này là snapshot trình bày pipeline hiện tại của codebase. Các file source được copy nguyên văn vào markdown code blocks để đọc/trình bày, không nhằm execute toàn bộ notebook từ trên xuống.


## High-level data flow

```text
JSON batch
  -> PredictionRequest
  -> TaskRouter
  -> Type 1: LLM/heuristic translation -> Logic IR -> KB -> ForwardChainSolver -> PredictionResponse
  -> Type 2: physics pipeline -> PredictionResponse
  -> official response + route reason -> output JSON
```

Useful commands:

```bash
python src/exact/scripts/run_predictions.py --no-llm --limit 5 --output outputs/logic/pred.json
EXACT_LLM_PROVIDER=local EXACT_LLM_MODEL=Qwen/Qwen2.5-1.5B-Instruct python src/exact/scripts/run_predictions.py --limit 1
```


## 1. Entry point: batch prediction runner

Script CLI đọc batch JSON, build `PredictionRequest`, route task, gọi pipeline theo type, rồi ghi predictions.


### `src/exact/scripts/run_predictions.py`

```python
#!/usr/bin/env python
"""Run the full EXACT prediction flow over a JSON batch.

This mirrors the production shape:
input instance -> PredictionRequest -> TaskRouter -> Type-specific pipeline.
"""

from __future__ import annotations

import argparse
import json
import logging
import sys
from pathlib import Path
from typing import Any

if __package__ in {None, ""}:
    sys.path.insert(0, str(Path(__file__).resolve().parents[2]))

from exact.config import get_settings
from exact.datasets.schemas import PredictionRequest, TaskType, to_official_response
from exact.llm_client import build_json_client_from_settings
from exact.logger import setup_logging
from exact.logic.translation.llm_translator import JsonLLMClient
from exact.logic.pipeline import run_type1_pipeline
from exact.router.task_router import TaskRouter
from exact.type2.pipeline import run_type2_pipeline


DEFAULT_INPUT = Path("src/exact/datasets/exact/Logic_Based_Educational_Queries_inference.json")
DEFAULT_OUTPUT = Path("artifacts/predictions/predictions.json")


def main() -> None:
    args = parse_args()
    if args.no_llm and args.require_llm:
        raise ValueError("--no-llm and --require-llm cannot be used together")

    instances = load_instances(args.input)
    if args.limit is not None:
        instances = instances[: args.limit]

    router = TaskRouter()
    settings = get_settings()
    setup_logging(
        level=settings.log_level,
        log_file=args.log_file,
        json_logs=settings.json_logs,
    )
    logger = logging.getLogger(__name__)

    translator_client: JsonLLMClient | None = (
        None if args.no_llm else build_json_client_from_settings(settings)
    )
    logger.info(
        "prediction runner: "
        f"provider={settings.llm_provider}, "
        f"model={settings.llm_model}, "
        f"llm_enabled={translator_client is not None}, "
        f"require_llm={args.require_llm}, "
        f"max_tokens={settings.llm_max_tokens}, "
        f"instances={len(instances)}"
    )
    predictions: list[dict[str, Any]] = []

    for index, instance in enumerate(instances, start=1):
        request = PredictionRequest.model_validate(instance)
        route = router.route(request)
        logger.info(
            "processing %s/%s id=%s route=%s",
            index,
            len(instances),
            request.id,
            route.task_type.value,
        )

        if route.task_type == TaskType.TYPE1_LOGIC:
            response = run_type1_pipeline(
                request,
                translator_client=translator_client,
                settings=settings,
                allow_heuristic_fallback=not args.require_llm,
            )
        elif route.task_type == TaskType.TYPE2_PHYSICS:
            response = run_type2_pipeline(request)
        else:
            raise ValueError(f"Unsupported task type: {route.task_type}")

        prediction = response.model_dump(mode="json")
        prediction["route_reason"] = route.reason
        prediction["official"] = to_official_response(response)
        predictions.append(prediction)

        if args.progress_every and index % args.progress_every == 0:
            status = f"processed {index}/{len(instances)} answer={response.answer}"
            if response.error:
                status += f" error={_shorten(response.error)}"
            logger.info(status)

    output = {
        "source": str(args.input),
        "count": len(predictions),
        "format": "exact_predictions",
        "predictions": predictions,
    }

    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(output, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    logger.info("wrote %s predictions to %s", len(predictions), args.output)


def load_instances(path: Path) -> list[dict[str, Any]]:
    payload = json.loads(path.read_text(encoding="utf-8"))

    if isinstance(payload, list):
        instances = payload
    elif isinstance(payload, dict) and isinstance(payload.get("instances"), list):
        instances = payload["instances"]
    else:
        raise ValueError(f"Expected a list or a top-level 'instances' list in {path}")

    if not all(isinstance(instance, dict) for instance in instances):
        raise ValueError(f"All instances in {path} must be JSON objects")

    return instances


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--input", type=Path, default=DEFAULT_INPUT)
    parser.add_argument("--output", type=Path, default=DEFAULT_OUTPUT)
    parser.add_argument("--limit", type=int, default=None)
    parser.add_argument("--progress-every", type=int, default=100)
    parser.add_argument("--log-file", type=Path, default=Path("outputs/logs/run_predictions.log"))
    parser.add_argument("--no-llm", action="store_true", help="Disable LLM translation fallback.")
    parser.add_argument(
        "--require-llm",
        action="store_true",
        help="Fail if LLM translation is unavailable or invalid; do not use heuristic parser.",
    )
    return parser.parse_args()


def _shorten(text: str, max_len: int = 180) -> str:
    text = " ".join(text.split())
    if len(text) <= max_len:
        return text
    return text[:max_len] + "..."


if __name__ == "__main__":
    main()

```


## 2. Runtime configuration and logging

`Settings` đọc `.env`/environment với prefix `EXACT_`; `logger.py` cấu hình log console/file và request context.


### `src/exact/config.py`

```python
from __future__ import annotations

from functools import lru_cache
from pathlib import Path
from typing import Literal

from pydantic import AliasChoices, Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

ROOT_DIR = Path(__file__).resolve().parents[2]
SRC_DIR = ROOT_DIR / "src"
PACKAGE_DIR = SRC_DIR / "exact"


class Settings(BaseSettings):
    """Application settings loaded from environment variables."""

    model_config = SettingsConfigDict(
        env_file=ROOT_DIR / ".env",
        env_file_encoding="utf-8",
        env_prefix="EXACT_",
        extra="ignore",
    )

    environment: Literal["local", "dev", "test", "prod"] = "local"

    data_dir: Path = PACKAGE_DIR / "datasets"
    exact_dataset_dir: Path = PACKAGE_DIR / "datasets" / "exact"
    type1_path: Path = exact_dataset_dir / "Logic_Based_Educational_Queries.json"
    type2_path: Path = exact_dataset_dir / "Physics_Problems_Text_Only.csv"

    artifacts_dir: Path = ROOT_DIR / "artifacts"
    predictions_dir: Path = artifacts_dir / "predictions"
    reports_dir: Path = artifacts_dir / "reports"
    splits_dir: Path = artifacts_dir / "splits"
    normalized_data_path: Path = artifacts_dir / "normalized_dataset.jsonl"

    split_ratios: dict[str, float] = Field(
        default_factory=lambda: {"train": 0.70, "dev": 0.15, "test": 0.15}
    )
    default_seed: int = 42

    llm_provider: Literal["openai", "anthropic", "local"] = "local"
    llm_model: str = Field(
        default="Qwen/Qwen2.5-7B-Instruct",
        validation_alias=AliasChoices("EXACT_LLM_MODEL", "EXACT_MODEL_ID"),
    )
    math_model_id: str = "Qwen/Qwen2.5-Math-7B-Instruct"
    llm_base_url: str | None = None
    llm_api_key: SecretStr | None = None
    mock_llm: bool = Field(
        default=False,
        validation_alias=AliasChoices("EXACT_MOCK_LLM", "MOCK_LLM"),
    )
    llm_max_tokens: int = Field(default=2048, ge=1, validation_alias="EXACT_MAX_NEW_TOKENS")
    llm_temperature: float = Field(
        default=0.0,
        ge=0.0,
        le=2.0,
        validation_alias=AliasChoices("EXACT_LLM_TEMPERATURE", "EXACT_TEMPERATURE"),
    )
    llm_top_p: float = Field(
        default=1.0,
        gt=0.0,
        le=1.0,
        validation_alias=AliasChoices("EXACT_LLM_TOP_P", "EXACT_TOP_P"),
    )
    llm_timeout_seconds: float = Field(default=30.0, gt=0)
    llm_max_retries: int = Field(default=3, ge=0, validation_alias="EXACT_MAX_RETRIES")

    api_host: str = "0.0.0.0"
    api_port: int = Field(default=8080, ge=1, le=65535)
    log_level: Literal["DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL"] = "INFO"
    json_logs: bool = False

    def ensure_artifact_dirs(self) -> None:
        for directory in [
            self.artifacts_dir,
            self.predictions_dir,
            self.reports_dir,
            self.splits_dir,
        ]:
            directory.mkdir(parents=True, exist_ok=True)


@lru_cache
def get_settings() -> Settings:
    return Settings()


settings = get_settings()
```


### `src/exact/logger.py`

```python
"""
Một module dùng để cấu hình và quản lý loggers trong dự án.

Author: Khang P. Nguyen
Date: 2026-05-22

Example:
    from exact.core.logger import setup_logging, get_request_logger

    setup_logging(level="INFO", log_file="outputs/logs/app.log")

    logger = get_request_logger(
        __name__,
        request_id="q001",
        task_type="type1_logic",
    )

    logger.info("Start pipeline")
    logger.warning("Parser failed, retrying with repair prompt")
    logger.error("Pipeline failed", exc_info=True)
"""
from __future__ import annotations

import logging
import sys
import json

from typing import Any
from datetime import datetime, timezone
from pathlib import Path

DEFAULT_LOG_FORMAT = (
    "%(asctime)s | %(levelname)-8s | %(name)s | "
    "request_id=%(request_id)s | task_type=%(task_type)s | %(message)s"
)

DEFAULT_DATE_FORMAT = "%Y-%m-%d %H:%M:%S"

class SafeExtraFormatter(logging.Formatter):
    '''
    A class used to format logs in a safe way. We can avoid errors when formatting
    log having fields 'extra' like request_id or task_type but some log line
    don't have these fields.

    Usage:
        logger = get_logger(__name__)
        logger.info("Hello World", extra={"request_id": "123", "task_type": "task1"})
        logger.info("Hello World")
    '''

    def format(self, record: logging.LogRecord) -> str:
        if not hasattr(record, "request_id"):
            record.request_id = "-"
        if not hasattr(record, "task_type"):
            record.task_type = "-"
        return super().format(record)

class JSONFormatter(logging.Formatter):
    """JSON formatter for production-freiendly structure logs"""

    def format(self, record: logging.LogRecord) -> str:
        '''
        A function is used to format log records into JSON format.
        
        Args:
            record (logging.LogRecord): The log record to format.
            
        Returns:
            str: The formatted log record in JSON format.
        '''
        log_data: dict[str, Any] = {
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "level": record.levelname,
            "logger": record.name,
            "message": record.getMessage(),
            "request_id": getattr(record, "request_id", "-"),
            "task_type": getattr(record, "task_type", "-"),
            "module": record.module,
            "function": record.funcName,
            "line": record.lineno,
        }

        if record.exc_info:
            log_data["exception"] = self.formatException(record.exc_info)

        return json.dumps(log_data, ensure_ascii=False)

def setup_logging(
    level: str = "INFO",
    log_file: str | Path | None = None,
    json_logs: bool = False,
) -> None:
    """
    Configure logging for the whole application

    Args:
        - level (str): The logging level (e.g., "DEBUG", "INFO", "WARNING", "ERROR", "CRITICAL").
        - log_file (str | Path | None): Optional path to a log file.
        - json_logs (bool): If True, logs will be in JSON format.
    """

    numeric_level = getattr(logging, level.upper(), logging.INFO)

    if json_logs:
        formatter: logging.Formatter = JSONFormatter()
    else:
        formatter = SafeExtraFormatter(
            fmt=DEFAULT_LOG_FORMAT,
            datefmt=DEFAULT_DATE_FORMAT,
        )

    # Create handlers based on configuration
    # Why: Handle logging output to different sinks (console, files, etc.)
    #       For example, console logger and file logger can have different formatters   
    #       And handlers can be enabled/disabled independently
    
    handlers: list[logging.Handler] = []

    # Console handler - always present
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setFormatter(formatter)
    handlers.append(console_handler)

    # Kiểm tra xem có log ra file hay không
    if log_file is not None:
        log_path = Path(log_file)
        log_path.parent.mkdir(parents=True, exist_ok=True)

        file_handler = logging.FileHandler(log_path, encoding="utf-8")
        file_handler.setFormatter(formatter)
        handlers.append(file_handler)
    
    # Lấy logger toàn cục, xóa toàn bộ các logger cục bộ của ứng dụng, reset lại cấu hình, đặt lại level với
    root_logger = logging.getLogger()
    root_logger.handlers.clear()
    root_logger.setLevel(numeric_level)

    for handler in handlers:
        root_logger.addHandler(handler)

    _quiet_noisy_loggers()

def _quiet_noisy_loggers() -> None:
    """
    Dùng để giảm bớt các log rác từ các thư viện bên ngoài như httpx, aiohttp...

    Trong project này, ta dùng rất nhiều thư viện như FastAPI / Uvicorn, httpx / request các thư viện này có thể tự in ra rất nhiều log
    nên ta cần dùng để có thể xóa bớt và chặn đi các log đó, tránh việc terminal hoặc file bị quá tải.
    """

    noisy_loggers = [
        "httpx",
        "urllib3",
        "transformers",
        "asyncio",
        "uvicorn.access",
    ]

    for logger_name in noisy_loggers:
        logging.getLogger(logger_name).setLevel(logging.WARNING)

def truncate_text(text: str | None, max_len: int = 300) -> str:
    """Safely truncate long text before logging."""
    if text is None:
        return ""
    if len(text) <= max_len:
        return text
    return text[:max_len] + "...[truncated]"

def get_logger(name: str) -> logging.Logger:
    """
    Get a logger instance
    Args:
        name (str): The name of the logger (usually __name__)
    Returns:
        logging.Logger: The logger instance
    """
    return logging.getLogger(name)

def get_request_logger(
    name: str,
    request_id: str | None = None,
    task_type: str | None = None,
    **extra: Any
) -> logging.LoggerAdapter:
    """
    Trả về một logger với bối cảnh của request-level .

    Ví dụ:
        logger = get_request_logger(
        "__name__, 
        request_id="q001", 
        task_type="type1_logic")
    """

    context : dict[str, Any] = {
        "request_id": request_id or "-",
        "task_type": task_type or "-",
    }

    return logging.LoggerAdapter(logging.getLogger(name), context)
```


## 3. Dataset schema and official response format

Schema chuẩn hóa input/output: `PredictionRequest`, `PredictionResponse`, task/question types, và format official.


### `src/exact/datasets/schemas.py`

```python
"""
Schemas for EXACT dataset

Author: Khang P. Nguyen
Date: 2026-05-22

Example:
    payload = {
    "id": "q001",
    "premises-NL": [
        "If a student has high GPA, the student is eligible for scholarship.",
        "An has high GPA."
    ],
    "question": "Is An eligible for scholarship?",
    "difficulty": "easy",
    }

    request = PredictionRequest.model_validate(payload)

    print(request.inferred_task_type)
    print(request.premises_nl)
    """

from __future__ import annotations

from typing import Any, Literal
from pydantic import BaseModel, Field, ConfigDict, field_validator
from enum import Enum

class Input(BaseModel):
    """Base Input Schema for both of types of documents"""
    id: str | None = Field(default=None, description="Document ID")
    question: str | None = Field(default=None, description="Question text")
    premises_nl : list[str] | None = Field(default=None, description="Premises list in natural language format")
    
    # Cho phép dùng alias lẫn tên field Python đều được.
    model_config = {
        "populate_by_name": True,
        "extra": "allow"
    }

class TaskType(str, Enum):
    TYPE1_LOGIC = "type1_logic"
    TYPE2_PHYSICS = "type2_physics"

    UNKNOWN = "unknown"

class QuestionType(str, Enum):
    MCQ = "mcq"
    YES_NO_UNCERTAIN = "yes_no_uncertain"
    OPEN_ENDED = "open_ended"
    NUMERICAL = "numerical"

    UNKNOWN = "unknown"

class AppBaseModel(BaseModel):
    """
    Strict base model for internal objects and outbound responses.
    Internal data should be controlled, so unknown fields are forbidden.
    """

    model_config = ConfigDict(
        populate_by_name=True,
        extra="forbid",
        str_strip_whitespace=True,
    )


class InboundBaseModel(BaseModel):
    """
    Lenient base model for external inputs.

    Organizer/dataset payloads may contain metadata fields we do not use yet,
    so extra fields are allowed at the boundary.
    """

    model_config = ConfigDict(
        populate_by_name=True,
        extra="allow",
        str_strip_whitespace=True,
    )


class PredictionRequest(InboundBaseModel):
    """
    Input sample sent to our API or loaded from dataset.

    Type 1:
        premises-NL + question

    Type 2:
        question only
    """

    id: str | None = None
    question: str
    premises_nl: list[str] | None = Field(default=None, alias="premises-NL")

    @field_validator("question")
    @classmethod
    def question_must_not_be_empty(cls, value: str) -> str:
        if not value.strip():
            raise ValueError("question must not be empty")
        return value

    @property
    def inferred_task_type(self) -> TaskType:
        if self.premises_nl:
            return TaskType.TYPE1_LOGIC
        return TaskType.TYPE2_PHYSICS

class BatchPredictionRequest(InboundBaseModel):
    instances: list[PredictionRequest]

    @field_validator("instances")
    @classmethod
    def instances_must_not_be_empty(
        cls,
        value: list[PredictionRequest],
    ) -> list[PredictionRequest]:
        if not value:
            raise ValueError("instances must not be empty")
        return value


class ErrorInfo(AppBaseModel):
    code: str
    message: str
    recoverable: bool = False


class ProofStep(AppBaseModel):
    step_id: str
    derived: str
    rule_id: str | None = None
    support: list[str] = Field(default_factory=list)
    natural_language: str | None = None


class Type1Evidence(AppBaseModel):
    solver: str
    label: Literal["entailed", "contradicted", "unknown"]
    premises_used: list[str] = Field(default_factory=list)
    proof_trace: list[ProofStep] = Field(default_factory=list)


class EquationStep(AppBaseModel):
    step_id: str
    expression: str
    result: str | None = None
    unit: str | None = None
    natural_language: str | None = None


class Type2Evidence(AppBaseModel):
    executor: str
    formula: str | None = None
    given: dict[str, Any] = Field(default_factory=dict)
    unit_conversions: list[str] = Field(default_factory=list)
    equation_trace: list[EquationStep] = Field(default_factory=list)


class PredictionResponse(AppBaseModel):
    """Competition response plus local metadata used during development."""

    # EXACT mandatory
    answer: str
    explanation: str

    # EXACT optional but encouraged
    fol: str | None = None
    cot: list[str] | None = None
    premises: list[str] | None = None
    confidence: float | None = Field(default=None, ge=0.0, le=1.0)

    # local/internal fields
    id: str | None = None
    task_type: TaskType | None = None
    question_type: QuestionType = QuestionType.UNKNOWN
    unit: str | None = None
    error: str | None = None


class BatchPredictionResponse(AppBaseModel):
    predictions: list[PredictionResponse]


def to_official_response(response: PredictionResponse) -> dict[str, Any]:
    """Convert an internal response into the stable EXACT response shape."""
    return {
        "answer": response.answer,
        "explanation": response.explanation,
        "fol": response.fol,
        "cot": response.cot,
        "premises": response.premises,
        "confidence": response.confidence,
    }

```


## 4. Task routing

`TaskRouter` quyết định request thuộc Type 1 logic hay Type 2 physics trước khi vào pipeline chuyên biệt.


### `src/exact/router/task_router.py`

```python
from dataclasses import dataclass

from exact.datasets.schemas import PredictionRequest, TaskType

@dataclass(frozen=True)
class RouteDecision:
    task_type: TaskType
    reason: str


# Backward-compatible alias for older imports.
RouterDecision = RouteDecision

class TaskRouter:
    def route(self, request: PredictionRequest) -> RouteDecision:
        if request.premises_nl:
            return RouteDecision(
                task_type=TaskType.TYPE1_LOGIC,
                reason="premises_nl present",
            )

        return RouteDecision(
            task_type=TaskType.TYPE2_PHYSICS,
            reason="premises_nl absent",
        )

```


## 5. Type 1 logic pipeline overview

Type 1 dùng LLM như semantic parser sang IR, sau đó reasoning bằng symbolic solver. Nếu không strict LLM thì fallback heuristic parser.


### `src/exact/logic/pipeline.py`

```python
"""Type 1 logic pipeline.

The default path is deterministic symbolic reasoning. An injected LLM client is
still supported for experiments and backwards-compatible tests, but production
code should treat that as parser/fallback plumbing rather than the core proof.
"""

from __future__ import annotations

import json
from typing import Any, Protocol

from exact.config import Settings, get_settings
from exact.datasets.schemas import PredictionRequest, PredictionResponse, QuestionType, TaskType
from exact.logic.explain import explain_result, kb_to_fol_like_text
from exact.logic.kb import build_kb_from_parsed_premises
from exact.logic.translation.llm_translator import JsonLLMClient
# Legacy walkthrough helper translate_with_fallback is defined in the translator section below.
from exact.logger import get_request_logger
from exact.symbolic_solvers import ForwardChainSolver


class SyncLLMClient(Protocol):
    def generate(self, prompt: str) -> str: ...


def run_type1_pipeline(
    request: PredictionRequest,
    llm_client: SyncLLMClient | None = None,
    translator_client: JsonLLMClient | None = None,
    settings: Settings | None = None,
    allow_heuristic_fallback: bool = True,
) -> PredictionResponse:
    """
    Answer a Type 1 logic query with symbolic proof where possible.

    Args:
        request (PredictionRequest): Đại diện cho request cần xử lý
        llm_client (SyncLLMClient): Đại diện cho LLMClient để gọi API
        translator_client (JsonLLMClient): Client dùng để dịch NL -> Logical Form
        settings (Settings): Các settings cho pipeline
        allow_heuristic_fallback (bool): Có dùng heuristic không

    Returns:
        PredictionResponse: Đại diện cho dự đoán
    """

    logger = get_request_logger(
        __name__,
        request_id=request.id,
        task_type=TaskType.TYPE1_LOGIC.value,
    )
    logger.info("Start Type 1 pipeline")

    if llm_client is not None:
        response = _run_injected_llm_path(request, llm_client)
        if response is not None:
            logger.info("Injected LLM response accepted")
            return response
        logger.info("Injected LLM response invalid; falling back to symbolic path")

    settings = settings or get_settings()
    premises = request.premises_nl or []
    parsed_premises, query, translation_warnings = translate_with_fallback(
        premises=premises,
        question=request.question,
        llm_client=translator_client,
        settings=settings,
        allow_heuristic_fallback=allow_heuristic_fallback,
    )
    kb = build_kb_from_parsed_premises(
        premises,
        parsed_premises,
        parser_version="llm_translator_v1" if settings.llm_base_url else "heuristic_horn_v1",
        extra_warnings=translation_warnings,
    )
    result = ForwardChainSolver().solve(kb, query.claim)
    explanation, cot, cited_premises = explain_result(result, kb)

    confidence = {
        "Yes": 0.78,
        "No": 0.76,
        "Unknown": 0.35,
    }[result.label]

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=QuestionType.YES_NO_UNCERTAIN,
        answer=result.label,
        explanation=explanation,
        fol=kb_to_fol_like_text(kb) or None,
        cot=cot,
        premises=cited_premises,
        confidence=confidence,
        error="; ".join(result.warnings) if result.warnings else None,
    )


def _run_injected_llm_path(
    request: PredictionRequest,
    llm_client: SyncLLMClient,
) -> PredictionResponse | None:
    prompt = _build_fallback_prompt(request)
    try:
        data = _parse_json_object(llm_client.generate(prompt))
    except (TypeError, ValueError, json.JSONDecodeError):
        return None

    answer = _normalize_answer(str(data.get("answer", "")))
    if not answer:
        return None

    confidence = data.get("confidence", 0.25)
    try:
        confidence = float(confidence)
    except (TypeError, ValueError):
        confidence = 0.25

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE1_LOGIC,
        question_type=QuestionType.YES_NO_UNCERTAIN,
        answer=answer,
        explanation=str(data.get("explanation") or "Answered by injected LLM fallback."),
        fol=data.get("fol"),
        cot=_as_string_list(data.get("cot")),
        premises=_as_string_list(data.get("premises")),
        confidence=max(0.0, min(1.0, confidence)),
        error=None,
    )


def _build_fallback_prompt(request: PredictionRequest) -> str:
    premises = "\n".join(
        f"P{idx + 1}: {premise}" for idx, premise in enumerate(request.premises_nl or [])
    )
    return (
        "Answer the logic question using only the premises. Return JSON with "
        "answer, explanation, fol, cot, premises, confidence.\n\n"
        f"{premises}\n\nQuestion: {request.question}"
    )


def _parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if not text:
        raise ValueError("empty LLM output")

    start = text.find("{")
    end = text.rfind("}")
    if start == -1 or end == -1 or end < start:
        raise ValueError("LLM output did not contain a JSON object")
    parsed = json.loads(text[start : end + 1])
    if not isinstance(parsed, dict):
        raise ValueError("LLM JSON output must be an object")
    return parsed


def _normalize_answer(answer: str) -> str:
    value = answer.strip().lower()
    if value in {"yes", "true", "entailed"}:
        return "Yes"
    if value in {"no", "false", "contradicted"}:
        return "No"
    if value in {"unknown", "uncertain", "not enough information"}:
        return "Unknown"
    if len(answer.strip()) == 1 and answer.strip().upper() in {"A", "B", "C", "D"}:
        return answer.strip().upper()
    return answer.strip()


def _as_string_list(value: Any) -> list[str] | None:
    if value is None:
        return None
    if isinstance(value, list):
        return [str(item) for item in value]
    return [str(value)]

```


## 6. LLM translator: natural language to Horn-style IR

`llm_translator.py` định nghĩa JSON schema LLM phải trả về, prompt translate, validate bằng Pydantic, rồi convert sang IR.


### `src/exact/logic/llm_translator.py`

```python
"""LLM-to-IR translator for Type 1 logic.

The LLM is used as a semantic parser, not as the final judge. It converts
natural-language premises/questions into the small IR consumed by deterministic
symbolic solvers, following the Logic-LM/LINC architecture.
"""

from __future__ import annotations

from typing import Any, Protocol

from openai.types.chat import ChatCompletionMessageParam
from pydantic import BaseModel, ConfigDict, Field, field_validator

from exact.config import Settings, get_settings
from exact.logic.ir import Atom, Fact, ParsedPremise, Query, Rule
from exact.logic.parsing.parser import atom_from_text, parse_premise_to_ir, parse_question_to_query
from exact.llm_client import LLMClient
from exact.logger import get_logger

logger = get_logger(__name__)


class JsonLLMClient(Protocol):
    def complete_json_sync(
        self,
        messages: list[ChatCompletionMessageParam],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict[str, Any]: ...


class AtomSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    text: str
    negated: bool = False

    @field_validator("text")
    @classmethod
    def text_must_not_be_empty(cls, value: str) -> str:
        value = value.strip()
        if not value:
            raise ValueError("atom text must not be empty")
        return value


class RuleSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    conditions: list[AtomSpec] = Field(default_factory=list)
    conclusion: AtomSpec

    @field_validator("conditions")
    @classmethod
    def conditions_must_not_be_empty(cls, value: list[AtomSpec]) -> list[AtomSpec]:
        if not value:
            raise ValueError("rule conditions must not be empty")
        return value


class PremiseSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    source_idx: int
    facts: list[AtomSpec] = Field(default_factory=list)
    rules: list[RuleSpec] = Field(default_factory=list)


class QuerySpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    claim: AtomSpec


class TranslationSpec(BaseModel):
    model_config = ConfigDict(extra="forbid")

    premises: list[PremiseSpec]
    query: QuerySpec


def translate_with_llm(
    premises: list[str],
    question: str,
    llm_client: JsonLLMClient | None = None,
    settings: Settings | None = None,
) -> tuple[tuple[ParsedPremise, ...], Query]:
    """Translate a Type 1 instance into IR using a real LLM client."""

    settings = settings or get_settings()
    client = llm_client or LLMClient.from_settings(settings)
    messages = _build_messages(premises, question)
    logger.info(
        "Starting LLM translation: premises=%s, question_chars=%s, max_tokens=%s",
        len(premises),
        len(question),
        settings.llm_max_tokens,
    )
    raw = client.complete_json_sync(
        messages=messages,
        temperature=settings.llm_temperature,
        max_tokens=settings.llm_max_tokens,
    )
    logger.info("Validating LLM translation schema")
    spec = TranslationSpec.model_validate(raw)
    logger.info("Converting LLM translation to IR")
    return _spec_to_ir(spec, premises, question)


def translate_with_fallback(
    premises: list[str],
    question: str,
    llm_client: JsonLLMClient | None = None,
    settings: Settings | None = None,
    allow_heuristic_fallback: bool = True,
) -> tuple[tuple[ParsedPremise, ...], Query, tuple[str, ...]]:
    """Try LLM translation, then fall back to the local parser with warnings."""

    settings = settings or get_settings()
    if llm_client is not None or settings.llm_provider == "local" or settings.llm_base_url:
        try:
            parsed, query = translate_with_llm(premises, question, llm_client, settings)
            return parsed, query, ()
        except Exception as exc:
            if not allow_heuristic_fallback:
                raise RuntimeError(f"LLM translation failed and fallback is disabled: {exc}") from exc
            warnings = (f"LLM translation failed; heuristic parser used: {exc}",)
            return _heuristic_translation(premises, question, warnings)

    if not allow_heuristic_fallback:
        raise RuntimeError("No LLM client configured and fallback is disabled")

    return _heuristic_translation(
        premises,
        question,
        ("No LLM client configured; heuristic parser used.",),
    )


def _heuristic_translation(
    premises: list[str],
    question: str,
    warnings: tuple[str, ...] = (),
) -> tuple[tuple[ParsedPremise, ...], Query, tuple[str, ...]]:
    parsed = tuple(parse_premise_to_ir(premise, idx) for idx, premise in enumerate(premises))
    query = parse_question_to_query(question)
    return parsed, query, warnings


def _build_messages(premises: list[str], question: str) -> list[ChatCompletionMessageParam]:
    premise_text = "\n".join(f"{idx}: {premise}" for idx, premise in enumerate(premises))
    return [
        {
            "role": "system",
            "content": (
                "You are a semantic parser for an educational logic QA system. "
                "Translate natural-language premises and the question into a compact Horn-style JSON IR. "
                "Do not answer the question. Use only the given text."
            ),
        },
        {
            "role": "user",
            "content": (
                "Return exactly this JSON shape:\n"
                "{\n"
                '  "premises": [\n'
                '    {"source_idx": 0, "facts": [{"text": "A", "negated": false}], '
                '"rules": [{"conditions": [{"text": "A"}], "conclusion": {"text": "B"}}]}\n'
                "  ],\n"
                '  "query": {"claim": {"text": "B", "negated": false}}\n'
                "}\n\n"
                "Rules:\n"
                "- source_idx must match the premise number shown below.\n"
                "- Use facts for directly stated atomic statements.\n"
                "- Use rules for if/then, who/that/when conditional rules, requirements, implications.\n"
                "- Split conjunctions into multiple condition atoms.\n"
                "- Preserve entities and predicates in simple English text.\n"
                "- Mark negated=true only for explicit negation.\n\n"
                f"Premises:\n{premise_text}\n\nQuestion:\n{question}"
            ),
        },
    ]


def _spec_to_ir(
    spec: TranslationSpec,
    raw_premises: list[str],
    raw_question: str,
) -> tuple[tuple[ParsedPremise, ...], Query]:
    parsed_by_idx: dict[int, ParsedPremise] = {}

    for premise_spec in spec.premises:
        source_idx = premise_spec.source_idx
        if source_idx < 0 or source_idx >= len(raw_premises):
            continue

        facts = tuple(
            Fact(
                atom=_atom_from_spec(atom_spec),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for atom_spec in premise_spec.facts
        )
        rules = tuple(
            Rule(
                conditions=tuple(_atom_from_spec(atom_spec) for atom_spec in rule_spec.conditions),
                conclusion=_atom_from_spec(rule_spec.conclusion),
                source_idx=source_idx,
                text=raw_premises[source_idx],
            )
            for rule_spec in premise_spec.rules
        )
        parsed_by_idx[source_idx] = ParsedPremise(facts=facts, rules=rules)

    parsed = []
    for source_idx, premise in enumerate(raw_premises):
        parsed.append(parsed_by_idx.get(source_idx) or ParsedPremise(warnings=(f"No LLM IR for premise {source_idx}",)))

    return tuple(parsed), Query(claim=_atom_from_spec(spec.query.claim), raw_question=raw_question)


def _atom_from_spec(spec: AtomSpec) -> Atom:
    atom = atom_from_text(spec.text)
    return Atom(pred=atom.pred, args=atom.args, negated=spec.negated, text=atom.text)

```


## 7. LLM clients: API-compatible and local HuggingFace

`LLMClient` gọi OpenAI-compatible APIs; `LocalClient` chạy model local bằng transformers; `LocalJsonClient` parse JSON output.


### `src/exact/llm_client.py`

```python
"""
Một module cung cấp một lớp để có thể tạo một LLMClient dùng để sinh các câu trả lời từ LLM
"""
from __future__ import annotations

import asyncio
import importlib.util
import json
import time
from typing import Any, Iterable
from openai import AsyncOpenAI
from openai.types.chat import ChatCompletionMessageParam
from pydantic import BaseModel, ValidationError
from transformers import AutoTokenizer, AutoModelForCausalLM

import torch

from exact.config import Settings, get_settings
from exact.logger import get_logger

logger = get_logger(__name__)

class LLMClient:
    """
    Một lớp duy nhất tạo LLMClient để gọi các API.

    Args:
        api_key: Khóa API để xác thực với dịch vụ LLM.
        base_url: URL cơ sở của dịch vụ LLM.
        model: Tên mô hình LLM để sử dụng.
        timeout: Thời gian chờ tối đa cho mỗi yêu cầu.
        max_retries: Số lần thử lại tối đa khi yêu cầu thất bại.
    """
    def __init__(
        self,
        api_key: str,
        base_url: str | None = None,
        model: str = "gpt-4o-mini",
        timeout: float = 60.0,
        max_retries: int = 2,
    ):
        self.model = model
        self.client = AsyncOpenAI(
            api_key=api_key,
            base_url=base_url,
            timeout=timeout,
            max_retries=max_retries,
        )

    async def complete_json(
        self,
        messages: Iterable[ChatCompletionMessageParam],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict[str, Any]:
        """
        Gọi API của LLM để sinh ra một câu trả lời dưới dạng JSON.

        Args:
            messages: Danh sách các tin nhắn để gửi đến LLM.
            temperature: Nhiệt độ cho quá trình sinh.
            max_tokens: Số lượng token tối đa cho phép.

        Returns:
            Một dictionary chứa kết quả từ LLM.
        """
        response = await self.client.chat.completions.create(
            model=self.model,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
            response_format={"type": "json_object"},
        )

        choice = response.choices[0]
        text = choice.message.content or ""

        try:
            return _parse_json_object(text)
        except ValueError as exc:
            raise ValueError(f"LLM returned invalid JSON with finish_reason={choice.finish_reason}: {text}") from exc

    async def complete_as(
        self,
        messages: Iterable[ChatCompletionMessageParam],
        schema: type[BaseModel],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> BaseModel:
        """
        Gọi API của LLM để sinh ra một câu trả lời và xác thực nó theo một schema Pydantic.
        """
        data = await self.complete_json(
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )

        try:
            return schema.model_validate(data)
        except ValidationError as exc:
            raise ValueError(f"LLM output does not match schema: {exc}") from exc

    def complete_json_sync(
        self,
        messages: Iterable[ChatCompletionMessageParam],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict[str, Any]:
        """
        Synchronous wrapper for command-line scripts and FastAPI sync routes.
        """
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return asyncio.run(
                self.complete_json(
                    messages=messages,
                    temperature=temperature,
                    max_tokens=max_tokens,
                )
            )
        raise RuntimeError("complete_json_sync cannot run inside an active event loop")

    @classmethod
    def from_settings(cls, settings: Settings | None = None) -> "LLMClient":
        settings = settings or get_settings()
        api_key = settings.llm_api_key.get_secret_value() if settings.llm_api_key else "EMPTY"
        return cls(
            api_key=api_key,
            base_url=settings.llm_base_url,
            model=settings.llm_model,
            timeout=settings.llm_timeout_seconds,
            max_retries=settings.llm_max_retries,
        )
        


class LocalClient:
    def __init__(self, model_name: str):
        logger.info("Loading local tokenizer for %s", model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        model_kwargs: dict[str, Any] = {"dtype": _preferred_torch_dtype()}
        if importlib.util.find_spec("accelerate") is not None:
            model_kwargs["device_map"] = "auto"

        logger.info("Loading local model %s with %s", model_name, model_kwargs)
        started_at = time.monotonic()
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs)
        if "device_map" not in model_kwargs and torch.cuda.is_available():
            self.model = self.model.to("cuda")
        logger.info("Loaded local model %s in %.1fs", model_name, time.monotonic() - started_at)

    def complete(self, messages: list[dict], max_new_tokens: int = 2048) -> str:
        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        logger.info("Tokenizing local LLM prompt")
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        input_length = inputs["input_ids"].shape[1]
        logger.info("Starting local generation: input_tokens=%s, max_new_tokens=%s", input_length, max_new_tokens)
        started_at = time.monotonic()

        outputs = self.model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=self.tokenizer.eos_token_id,
        )

        generated_tokens = outputs[0][input_length:]
        logger.info(
            "Finished local generation: output_tokens=%s, elapsed=%.1fs",
            len(generated_tokens),
            time.monotonic() - started_at,
        )
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)


class LocalJsonClient:
    """JSON adapter over the direct transformers-based LocalClient."""

    def __init__(self, model_name: str):
        self.client = LocalClient(model_name)

    def complete_json_sync(
        self,
        messages: Iterable[ChatCompletionMessageParam],
        temperature: float = 0.0,
        max_tokens: int = 2048,
    ) -> dict[str, Any]:
        text = self.client.complete(
            messages=list(messages),
            max_new_tokens=max_tokens,
        )
        return _parse_json_object(text)


def build_json_client_from_settings(settings: Settings | None = None) -> Any | None:
    """Build a JSON-producing LLM client from runtime settings.

    - `llm_provider=local`: load the model directly with transformers.
    - `llm_base_url` set: call an OpenAI-compatible local/remote server.
    - otherwise: return None and let the pipeline use heuristic fallback.
    """

    settings = settings or get_settings()
    if settings.llm_provider == "local":
        return LocalJsonClient(settings.llm_model)
    if settings.llm_base_url:
        return LLMClient.from_settings(settings)
    return None


def _parse_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    start = text.find("{")
    end = text.rfind("}")
    if start == -1:
        raise ValueError(f"LLM output did not contain a JSON object: {text}")
    if end == -1 or end < start:
        raise ValueError(f"LLM output contained incomplete JSON; increase EXACT_MAX_NEW_TOKENS: {text}")
    try:
        parsed = json.loads(text[start : end + 1])
    except json.JSONDecodeError as exc:
        raise ValueError(f"LLM returned invalid JSON: {text}") from exc
    if not isinstance(parsed, dict):
        raise ValueError("LLM JSON output must be an object")
    return parsed


def _preferred_torch_dtype() -> torch.dtype:
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32

```


## 8. Logic IR and heuristic parser

IR là biểu diễn trung gian gồm atom/fact/rule/query. Parser heuristic xử lý khi không có LLM hoặc LLM fail mà fallback được bật.


### `src/exact/logic/ir.py`

```python
"""
Internal representations for the Type 1 logic branch.

The first production target is a small, inspectable Horn-style core:
LLM/CLOVER-style parsers can emit these objects, VERUS-style KB caching can
reuse them, and a Logic-LM/LINC-style solver/verifier can reason over them.
"""

from __future__ import annotations

from dataclasses import dataclass, field


@dataclass(frozen=True, order=True)
class Atom:
    """A normalized propositional or predicate-like logical statement."""

    pred: str
    args: tuple[str, ...] = ()
    negated: bool = False
    text: str | None = None

    def positive(self) -> "Atom":
        return Atom(pred=self.pred, args=self.args, negated=False, text=self.text)

    def negation(self) -> "Atom":
        return Atom(pred=self.pred, args=self.args, negated=not self.negated, text=self.text)

    def display(self) -> str:
        label = self.text or (
            f"{self.pred}({', '.join(self.args)})" if self.args else self.pred.replace("_", " ")
        )
        return f"not {label}" if self.negated else label


@dataclass(frozen=True)
class Rule:
    """A Horn-style rule: all conditions must hold to derive conclusion."""

    conditions: tuple[Atom, ...]
    conclusion: Atom
    source_idx: int
    text: str


@dataclass(frozen=True)
class Fact:
    """A directly stated premise fact."""

    atom: Atom
    source_idx: int
    text: str


@dataclass(frozen=True)
class ProofStep:
    """One derivation step with provenance for explanation and scoring depth."""

    derived: Atom
    used_premises: tuple[int, ...]
    rule_idx: int | None
    parents: tuple[Atom, ...] = ()
    natural_language: str | None = None


@dataclass(frozen=True)
class ParsedPremise:
    """Parser output for one source premise."""

    facts: tuple[Fact, ...] = ()
    rules: tuple[Rule, ...] = ()
    warnings: tuple[str, ...] = ()


@dataclass(frozen=True)
class Query:
    """Normalized target claim for yes/no/unknown reasoning."""

    claim: Atom
    raw_question: str
    expects_negation: bool = False


@dataclass(frozen=True)
class SolveResult:
    """Result returned by a symbolic solver."""

    label: str
    claim: Atom
    proof: tuple[ProofStep, ...] = ()
    supporting_premises: tuple[int, ...] = ()
    mode: str = "symbolic_forward_chain"
    warnings: tuple[str, ...] = ()


@dataclass(frozen=True)
class Theory:
    """Future extension point for richer typed FOL/Z3 encodings."""

    sorts: dict[str, list[str]] = field(default_factory=dict)
    predicates: dict[str, tuple[str, ...]] = field(default_factory=dict)
    functions: dict[str, tuple[tuple[str, ...], str]] = field(default_factory=dict)
    constants: dict[str, str] = field(default_factory=dict)


@dataclass(frozen=True)
class PremiseItem:
    """Unified representation of an NL/FOL premise pair from training data."""

    id: str
    nl: str | None
    fol: str | None
    source: str


@dataclass(frozen=True)
class SymbolEvidence:
    """Traceable mapping from a symbol back to its source premise."""

    premise_id: str
    source: str
    symbol: str
    confidence: float = 1.0

```


### `src/exact/logic/parser.py`

```python
"""Lightweight Type 1 parser.

This is intentionally conservative: it handles common regulation-style Horn
patterns now, while leaving a clean replacement boundary for LLM/CLOVER-style
decomposition later.
"""

from __future__ import annotations

import re
import unicodedata

from exact.logic.ir import Atom, Fact, ParsedPremise, Query, Rule

_IF_THEN_RE = re.compile(r"^\s*if\s+(.+?)\s*,?\s+then\s+(.+?)\.?\s*$", re.IGNORECASE)
_TRAILING_PUNCT_RE = re.compile(r"[\s.?!:;]+$")
_WORD_RE = re.compile(r"[a-z0-9]+")


def parse_premise_to_ir(premise: str, source_idx: int) -> ParsedPremise:
    """Parse one natural-language premise into facts/rules.

    Current MVP supports:
    - "If A then B." -> Rule(A -> B)
    - "If A and B, then C." -> Rule(A & B -> C)
    - "A." -> Fact(A)
    """

    text = premise.strip()
    if not text:
        return ParsedPremise(warnings=(f"premise {source_idx + 1} is empty",))

    match = _IF_THEN_RE.match(text)
    if match:
        antecedent, consequent = match.groups()
        conditions = tuple(_atom_from_clause(part) for part in _split_conjunction(antecedent))
        conclusion = _atom_from_clause(consequent)
        return ParsedPremise(
            rules=(
                Rule(
                    conditions=conditions,
                    conclusion=conclusion,
                    source_idx=source_idx,
                    text=text,
                ),
            )
        )

    return ParsedPremise(facts=(Fact(atom=_atom_from_clause(text), source_idx=source_idx, text=text),))


def parse_question_to_query(question: str) -> Query:
    """Parse a yes/no-style question into a target claim."""

    raw = question.strip()
    normalized = _strip_question_shell(raw)
    atom = _atom_from_clause(normalized)
    return Query(claim=atom, raw_question=raw, expects_negation=atom.negated)


def atom_from_text(text: str) -> Atom:
    """Public helper for tests and future LLM parser adapters."""

    return _atom_from_clause(text)


def _split_conjunction(text: str) -> list[str]:
    parts = re.split(r"\s+(?:and|&)\s+", text, flags=re.IGNORECASE)
    return [part.strip() for part in parts if part.strip()]


def _atom_from_clause(text: str) -> Atom:
    clause = _clean_clause(text)
    negated = False

    for prefix in ("it is not true that ", "not ", "does not ", "do not ", "did not "):
        if clause.startswith(prefix):
            negated = True
            clause = clause[len(prefix) :].strip()
            break

    pred = _slugify(_canonicalize_clause(clause))
    if not pred:
        pred = "unknown"

    return Atom(pred=pred, negated=negated, text=clause)


def _strip_question_shell(question: str) -> str:
    text = _clean_clause(question)
    text = re.sub(
        r"^(?:based on the above premises,\s*)?(?:does|do|did|is|are|can|could|will|would|should)\s+",
        "",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(r"\s+(?:hold|holds|follow|follows)$", "", text, flags=re.IGNORECASE)
    return text.strip()


def _clean_clause(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.strip().strip("\"'")
    text = _TRAILING_PUNCT_RE.sub("", text)
    return re.sub(r"\s+", " ", text).lower()


def _canonicalize_clause(text: str) -> str:
    text = re.sub(r"^(?:a|an|the)\s+", "", text)
    text = re.sub(r"\s+(?:is|are|was|were)\s+true$", "", text)
    return text.strip()


def _slugify(text: str) -> str:
    return "_".join(_WORD_RE.findall(text))

```


## 9. Knowledge base construction and explanation

KB gom facts/rules/warnings để solver xử lý; explain module tạo explanation, chain-of-thought dạng tóm tắt, và cited premises.


### `src/exact/logic/kb.py`

```python
"""Knowledge-base construction and caching for Type 1 logic queries."""

from __future__ import annotations

import hashlib
from dataclasses import dataclass

from exact.logic.ir import Fact, Rule, Theory
from exact.logic.ir import ParsedPremise
from exact.logic.parsing.parser import parse_premise_to_ir

PARSER_VERSION = "heuristic_horn_v1"


@dataclass(frozen=True)
class KnowledgeBase:
    """Parsed, reusable premise set with source-preserving facts and rules."""

    raw_premises: tuple[str, ...]
    facts: tuple[Fact, ...]
    rules: tuple[Rule, ...]
    premise_hash: str
    parser_version: str = PARSER_VERSION
    theory: Theory | None = None
    warnings: tuple[str, ...] = ()


_KB_CACHE: dict[str, KnowledgeBase] = {}


def hash_premises(premises: list[str] | tuple[str, ...], parser_version: str = PARSER_VERSION) -> str:
    """Hash premises plus parser version so stale KBs do not leak across parser changes."""

    text = parser_version + "\n" + "\n".join(premises)
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def build_kb_from_premises(
    premises: list[str] | tuple[str, ...],
    premise_hash: str | None = None,
    parser_version: str = PARSER_VERSION,
) -> KnowledgeBase:
    """Build a KB once per premise set, following VERUS-LM's separation idea."""

    facts: list[Fact] = []
    rules: list[Rule] = []
    warnings: list[str] = []
    raw_premises = tuple(premises)

    for source_idx, premise in enumerate(raw_premises):
        parsed = parse_premise_to_ir(premise=premise, source_idx=source_idx)
        facts.extend(parsed.facts)
        rules.extend(parsed.rules)
        warnings.extend(parsed.warnings)

    return KnowledgeBase(
        raw_premises=raw_premises,
        facts=tuple(facts),
        rules=tuple(rules),
        premise_hash=premise_hash or hash_premises(raw_premises, parser_version),
        parser_version=parser_version,
        warnings=tuple(warnings),
    )


def build_kb_from_parsed_premises(
    premises: list[str] | tuple[str, ...],
    parsed_premises: tuple[ParsedPremise, ...],
    premise_hash: str | None = None,
    parser_version: str = PARSER_VERSION,
    extra_warnings: tuple[str, ...] = (),
) -> KnowledgeBase:
    facts: list[Fact] = []
    rules: list[Rule] = []
    warnings: list[str] = list(extra_warnings)
    raw_premises = tuple(premises)

    for parsed in parsed_premises:
        facts.extend(parsed.facts)
        rules.extend(parsed.rules)
        warnings.extend(parsed.warnings)

    return KnowledgeBase(
        raw_premises=raw_premises,
        facts=tuple(facts),
        rules=tuple(rules),
        premise_hash=premise_hash or hash_premises(raw_premises, parser_version),
        parser_version=parser_version,
        warnings=tuple(warnings),
    )


def get_or_build_kb(premises: list[str] | tuple[str, ...]) -> KnowledgeBase:
    key = hash_premises(premises)
    if key not in _KB_CACHE:
        _KB_CACHE[key] = build_kb_from_premises(premises, premise_hash=key)
    return _KB_CACHE[key]


def clear_kb_cache() -> None:
    _KB_CACHE.clear()

```


### `src/exact/logic/explain.py`

```python
"""Proof-trace to EXACT-facing explanation helpers."""

from __future__ import annotations

from exact.logic.ir import SolveResult
from exact.logic.kb import KnowledgeBase


def explain_result(result: SolveResult, kb: KnowledgeBase) -> tuple[str, list[str], list[str]]:
    """Return explanation, CoT-style trace, and cited premise labels."""

    premise_labels = [f"P{idx + 1}" for idx in result.supporting_premises]

    if result.label == "Unknown":
        return (
            "The provided premises do not prove the claim or its negation, so the answer is Unknown.",
            ["No symbolic proof was found for the claim or for its negation."],
            [],
        )

    cot = [step.natural_language or f"Derived {step.derived.display()}." for step in result.proof]
    support_text = ", ".join(premise_labels) if premise_labels else "the parsed premises"

    if result.label == "Yes":
        explanation = f"Using {support_text}, the symbolic proof derives {result.claim.display()}."
    else:
        explanation = (
            f"Using {support_text}, the symbolic proof derives the negation of "
            f"{result.claim.display()}."
        )

    return explanation, cot, premise_labels


def kb_to_fol_like_text(kb: KnowledgeBase) -> str:
    """Readable formalization for the optional `fol` field."""

    lines: list[str] = []
    for fact in kb.facts:
        lines.append(f"P{fact.source_idx + 1}: {fact.atom.display()}")
    for rule in kb.rules:
        conditions = " AND ".join(condition.display() for condition in rule.conditions)
        lines.append(f"P{rule.source_idx + 1}: {conditions} -> {rule.conclusion.display()}")
    return "\n".join(lines)

```


## 10. Symbolic solver path

Forward-chain solver suy diễn closure từ facts/rules và trả nhãn Yes/No/Unknown cùng proof/warnings.


### `src/exact/symbolic_solvers/base.py`

```python
"""Common interfaces for symbolic solver backends."""

from __future__ import annotations

from typing import Protocol

from exact.logic.ir import Atom, SolveResult
from exact.logic.kb import KnowledgeBase


class SymbolicSolver(Protocol):
    """Solver interface consumed by Type 1 pipelines."""

    name: str

    def solve(self, kb: KnowledgeBase, claim: Atom) -> SolveResult:
        """Return a symbolic judgment and proof trace for `claim`."""
        ...

```


### `src/exact/symbolic_solvers/forward_chain/solver.py`

```python
"""Forward-chaining Horn solver.

This backend is the first deterministic executor in the Logic-LM style stack.
It is deliberately small, explainable, and source-preserving; richer engines
such as Z3 can be added beside it without changing the pipeline contract.
"""

from __future__ import annotations

from dataclasses import dataclass

from exact.logic.ir import Atom, ProofStep, SolveResult
from exact.logic.kb import KnowledgeBase


@dataclass(frozen=True)
class ForwardChainSolver:
    """Derive all Horn consequences and answer by proof lookup."""

    name: str = "forward_chain_horn"

    def solve(self, kb: KnowledgeBase, claim: Atom) -> SolveResult:
        return solve_query(kb, claim, mode=self.name)


def solve_query(
    kb: KnowledgeBase,
    claim: Atom,
    mode: str = "forward_chain_horn",
) -> SolveResult:
    """Prove claim, prove its negation, or return Unknown."""

    known, proofs = derive_closure(kb)

    if claim in known:
        proof = _trace_proof(claim, proofs)
        return SolveResult(
            label="Yes",
            claim=claim,
            proof=tuple(proof),
            supporting_premises=_support_from_proof(proof),
            mode=mode,
            warnings=kb.warnings,
        )

    negated_claim = claim.negation()
    if negated_claim in known:
        proof = _trace_proof(negated_claim, proofs)
        return SolveResult(
            label="No",
            claim=claim,
            proof=tuple(proof),
            supporting_premises=_support_from_proof(proof),
            mode=mode,
            warnings=kb.warnings,
        )

    return SolveResult(
        label="Unknown",
        claim=claim,
        proof=(),
        supporting_premises=(),
        mode=mode,
        warnings=kb.warnings,
    )


def derive_closure(kb: KnowledgeBase) -> tuple[set[Atom], dict[Atom, ProofStep]]:
    """Derive all reachable atoms and keep the first proof for each atom."""

    known: set[Atom] = set()
    proofs: dict[Atom, ProofStep] = {}

    for fact in kb.facts:
        if fact.atom not in known:
            known.add(fact.atom)
            proofs[fact.atom] = ProofStep(
                derived=fact.atom,
                used_premises=(fact.source_idx,),
                rule_idx=None,
                parents=(),
                natural_language=f"Premise {fact.source_idx + 1} states {fact.atom.display()}.",
            )

    changed = True
    while changed:
        changed = False
        for rule in kb.rules:
            if rule.conclusion in known:
                continue
            if all(condition in known for condition in rule.conditions):
                known.add(rule.conclusion)
                parent_premises: list[int] = [rule.source_idx]
                for condition in rule.conditions:
                    parent_premises.extend(proofs[condition].used_premises)
                proofs[rule.conclusion] = ProofStep(
                    derived=rule.conclusion,
                    used_premises=tuple(sorted(set(parent_premises))),
                    rule_idx=rule.source_idx,
                    parents=rule.conditions,
                    natural_language=(
                        f"Premise {rule.source_idx + 1} derives {rule.conclusion.display()} "
                        f"when {', '.join(parent.display() for parent in rule.conditions)} hold."
                    ),
                )
                changed = True

    return known, proofs


def _trace_proof(target: Atom, proofs: dict[Atom, ProofStep]) -> list[ProofStep]:
    ordered: list[ProofStep] = []
    visited: set[Atom] = set()

    def visit(atom: Atom) -> None:
        if atom in visited or atom not in proofs:
            return
        visited.add(atom)
        for parent in proofs[atom].parents:
            visit(parent)
        ordered.append(proofs[atom])

    visit(target)
    return ordered


def _support_from_proof(proof: list[ProofStep]) -> tuple[int, ...]:
    support: set[int] = set()
    for step in proof:
        support.update(step.used_premises)
    return tuple(sorted(support))

```


## 11. Type 2 physics pipeline

Nhánh Type 2 hiện xử lý physics bằng pipeline riêng, không nằm trọng tâm logic pipeline nhưng vẫn đi qua runner/router chung.


### `src/exact/type2/pipeline.py`

```python
from __future__ import annotations

from exact.datasets.schemas import PredictionRequest, PredictionResponse, QuestionType, TaskType
from exact.logger import get_request_logger


TYPE2_NOT_IMPLEMENTED_MESSAGE = (
    "Type 2 physics reasoning is reserved for the dedicated physics pipeline. "
    "This placeholder keeps the API contract stable while the team implements "
    "paper-backed quantity extraction, formula selection, execution, and verification."
)


def run_type2_pipeline(request: PredictionRequest) -> PredictionResponse:
    """Stable API placeholder for future Type 2 physics work.

    A teammate can replace this function with the real Type 2 implementation.
    Keep the return type and fallback behavior so `/predict` and `/batch` never
    crash while research-driven modules are being developed.
    """
    logger = get_request_logger(
        __name__,
        request_id=request.id,
        task_type=TaskType.TYPE2_PHYSICS.value,
    )
    logger.info("Start Type 2 pipeline")
    logger.info("Type 2 placeholder response returned")

    return PredictionResponse(
        id=request.id,
        task_type=TaskType.TYPE2_PHYSICS,
        question_type=QuestionType.NUMERICAL,
        answer="",
        explanation=TYPE2_NOT_IMPLEMENTED_MESSAGE,
        fol=None,
        cot=[
            "The request was routed to the Type 2 physics branch.",
            "The production physics solver has not been implemented in this repo yet.",
        ],
        premises=[
            "Type 2 receives only the question text.",
            "A future physics pipeline should derive answers from formulas, units, and verified computation.",
        ],
        confidence=0.0,
        error="type2_pipeline_not_implemented",
    )

```


## 12. Optional API entrypoints

Các entrypoint FastAPI dùng cùng schema/router/pipeline để expose service thay vì batch CLI.


### `src/exact/app/main.py`

```python
from __future__ import annotations

from fastapi import FastAPI

from exact.app.router import api_router
from exact.logger import setup_logging


def create_app() -> FastAPI:
    setup_logging(level="INFO", log_file="outputs/logs/api.log")

    app = FastAPI(
        title="TraceQA EXACT 2026 API",
        version="0.1.0",
    )
    app.include_router(api_router)
    return app


app = create_app()

```


### `src/exact/app/router.py`

```python
from __future__ import annotations

from fastapi import APIRouter, Request

from exact.datasets.schemas import (
    BatchPredictionRequest,
    BatchPredictionResponse,
    PredictionRequest,
    PredictionResponse,
    TaskType,
)
from exact.logger import get_request_logger
from exact.router.task_router import TaskRouter
from exact.logic.pipeline import run_type1_pipeline
from exact.type2.pipeline import run_type2_pipeline

api_router = APIRouter()
task_router = TaskRouter()


@api_router.get("/health")
def health_check() -> dict[str, str]:
    return {"status": "ok"}


@api_router.post("/predict", response_model=PredictionResponse)
def predict(payload: PredictionRequest, request: Request) -> PredictionResponse:
    route = task_router.route(payload)
    logger = get_request_logger(
        __name__,
        request_id=payload.id or request.headers.get("X-Request-ID"),
        task_type=route.task_type.value,
    )

    logger.info("Received request")
    logger.info("Route decision: %s", route.reason)

    try:
        if route.task_type == TaskType.TYPE1_LOGIC:
            return run_type1_pipeline(payload)
        return run_type2_pipeline(payload)
    except Exception as exc:
        logger.error("Prediction failed", exc_info=True)
        return PredictionResponse(
            id=payload.id,
            task_type=route.task_type,
            answer="",
            explanation=f"Prediction failed: {exc}",
            fol=None,
            cot=["The system attempted to process the request but failed."],
            premises=[],
            confidence=0.0,
            error=str(exc),
        )


@api_router.post("/batch", response_model=BatchPredictionResponse)
def batch_predict(
    payload: BatchPredictionRequest,
    request: Request,
) -> BatchPredictionResponse:
    predictions = [predict(item, request) for item in payload.instances]
    return BatchPredictionResponse(predictions=predictions)

```
